AQUEST CODI ACONSEGUEIX BAIXAR LES DADES FUNADMENTALS DELS ANYS 2009 FINS 2014

In [2]:
# ---------- retreive finanacials from edgar years 2009 - 2014 ----------
import re, io, zipfile, time, requests, pandas as pd
from urllib.parse import urljoin
from lxml import etree
from datetime import datetime
from bs4 import BeautifulSoup

# apple cik 0000320193
# microsoft cik 0000789019
# Walmart cik 0000104169
# Tesla cik 0001318605


# -------- CONFIG --------
USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
PAUSE = 0.2
CIK = "0001318605" # change to your desired CIK
FORMS = {"10-K","10-Q"}

DATE_FROM = "2009-07-22"
DATE_TO   = "2017-03-01 "      #"2014-07-23"

USGAAP_TAGS = {
    "revenue": [
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues"
    ],
    "net_income": ["NetIncomeLoss"],
    "assets": ["Assets"],
    "liabilities": ["Liabilities"],
    "equity": [
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "StockholdersEquity"
    ]
}

# ------------------- HTTP helpers -------------------
def _get(url, timeout=60, tries=4, backoff=0.9):
    last_err = None
    for k in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code in (403, 429):
                time.sleep(backoff * (k + 1))
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            last_err = e
            time.sleep(backoff * (k + 1))
    raise last_err

def fetch_json(url, timeout=60):
    return _get(url, timeout=timeout).json()

def fetch_bytes(url, timeout=90):
    return _get(url, timeout=timeout).content

# ------------------- PART 1: Filings -------------------
def get_filings_json(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    j = fetch_json(sub_url)

    def pack_from_recent(recent_dict):
        return pd.DataFrame({
            "accession": recent_dict["accessionNumber"],
            "form": recent_dict["form"],
            "filed": recent_dict["filingDate"],
            "primary": recent_dict["primaryDocument"],
            "source": "json"
        })

    recent_df = pack_from_recent(j["filings"]["recent"])
    older_parts = []
    for f in j.get("filings", {}).get("files", []):
        url = f"https://data.sec.gov/submissions/{f['name']}"
        y = fetch_json(url)
        if "filings" in y and "recent" in y["filings"]:
            older_parts.append(pack_from_recent(y["filings"]["recent"]))
        time.sleep(PAUSE)

    all_filings = pd.concat([recent_df] + older_parts, ignore_index=True)
    all_filings["filed"] = pd.to_datetime(all_filings["filed"])
    return all_filings

def get_filings_atom(cik, form_type="10-K"):
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type={form_type}&owner=exclude&count=100&output=atom"
    r = _get(url, timeout=30)
    soup = BeautifulSoup(r.text, "xml")
    entries = []
    for entry in soup.find_all("entry"):
        filed = entry.find("filing-date").text if entry.find("filing-date") else None
        link = entry.find("link")["href"] if entry.find("link") else None
        acc = entry.find("accession-number").text if entry.find("accession-number") else None
        entries.append((acc, form_type, filed, link))
    return entries

def get_all_atom(cik):
    filings = []
    for form in FORMS:
        filings += get_filings_atom(cik, form)
        time.sleep(PAUSE)
    df = pd.DataFrame(filings, columns=["accession","form","filed","primary"])
    df["filed"] = pd.to_datetime(df["filed"])
    df["source"] = "atom"
    return df

def merge_sources(cik):
    df_json = get_filings_json(cik)
    df_atom = get_all_atom(cik)
    all_filings = pd.concat([df_json, df_atom], ignore_index=True)
    all_filings = all_filings.drop_duplicates(subset=["accession","form","filed"])
    all_filings = all_filings[all_filings["form"].isin(FORMS)]
    return all_filings.sort_values("filed")

# ------------------- PART 2: XBRL -------------------
def derive_base_dir(index_url: str) -> str:
    m = re.search(r"/data/(\d+)/(\d{10}-\d{2}-\d{6})(?:/|)", index_url)
    if m:
        cik = m.group(1)
        acc_no_dash = m.group(2).replace("-", "")
        return f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dash}/"
    return index_url.rsplit("/", 1)[0] + "/"

def candidates_from_indexjson(base_dir: str):
    exts_order = [".xml", ".xbrl", ".ins", ".zip", ".html", ".htm"]
    meta = fetch_json(base_dir + "index.json")
    items = meta.get("directory", {}).get("item", [])
    cands = []
    for it in items:
        name = it.get("name", "")
        low = name.lower()
        if any(low.endswith(ext) for ext in exts_order):
            if any(sfx in low for sfx in ["_cal", "_def", "_lab", "_pre", ".xsd"]):
                continue
            cands.append((exts_order.index(next(ext for ext in exts_order if low.endswith(ext))), it.get("size", 0), base_dir + name))
    cands.sort(key=lambda t: (t[0], -t[1]))
    return [u for _,__,u in cands]

def candidates_from_index_page(index_url: str):
    html = _get(index_url, timeout=60).text
    hrefs = re.findall(r'href="([^"]+)"', html, flags=re.I)
    hrefs = [urljoin(index_url, h) for h in hrefs]
    exts_order = [".xml", ".xbrl", ".ins", ".zip", ".html", ".htm"]
    cands = []
    for u in hrefs:
        low = u.lower()
        if any(low.endswith(ext) for ext in exts_order):
            if any(sfx in low for sfx in ["_cal", "_def", "_lab", "_pre", ".xsd"]):
                continue
            cands.append((exts_order.index(next(ext for ext in exts_order if low.endswith(ext))), u))
    cands.sort(key=lambda t: t[0])
    return [u for _,u in cands]

def sniff_instance_from_bytes(blob: bytes):
    try:
        parser = etree.XMLParser(recover=True, huge_tree=True)
        root = etree.fromstring(blob, parser=parser)
    except Exception:
        return None
    if root.find(".//{*}context") is None:
        return None
    return root

def find_xbrl_instance_and_root(index_url: str):
    base = derive_base_dir(index_url)
    def candidate_streams():
        try:
            for u in candidates_from_indexjson(base):
                yield u
        except Exception: pass
        try:
            for u in candidates_from_index_page(index_url):
                yield u
        except Exception: pass

    for url in candidate_streams():
        try:
            if url.lower().endswith(".zip"):
                blob = fetch_bytes(url)
                with zipfile.ZipFile(io.BytesIO(blob)) as zf:
                    names = [(zi.filename, zi.file_size) for zi in zf.infolist()
                             if re.search(r"\.(xml|xbrl|ins)$", zi.filename, re.I)
                             and not re.search(r"(_cal|_def|_lab|_pre|\.xsd)$", zi.filename, re.I)]
                    names.sort(key=lambda t: -t[1])
                    for name,_ in names:
                        root = sniff_instance_from_bytes(zf.read(name))
                        if root is not None:
                            return (f"{url}::{name}", root)
            else:
                blob = fetch_bytes(url)
                root = sniff_instance_from_bytes(blob)
                if root is not None:
                    return (url, root)
        except Exception:
            continue
    raise RuntimeError("No usable XBRL instance found")

# ---------- PARSER with period_end ----------
def parse_xbrl_root(root: etree._Element, form_type: str):
    ns = root.nsmap
    contexts = {}
    for ctx in root.findall(".//{*}context", namespaces=ns):
        cid = ctx.get("id")
        if not cid: continue
        start = ctx.find(".//{*}startDate", namespaces=ns)
        end   = ctx.find(".//{*}endDate", namespaces=ns)
        inst  = ctx.find(".//{*}instant", namespaces=ns)

        if inst is not None and (inst.text or "").strip():
            try:
                contexts[cid] = ("instant", datetime.strptime(inst.text[:10], "%Y-%m-%d"), None)
            except: pass
        elif start is not None and end is not None:
            try:
                sdt = datetime.strptime(start.text[:10], "%Y-%m-%d")
                edt = datetime.strptime(end.text[:10], "%Y-%m-%d")
                delta = (edt - sdt).days
                contexts[cid] = ("duration", edt, delta)
            except: pass

    def best_fact(local, want):
        elems = root.findall(f".//{{*}}{local}", namespaces=ns)
        chosen_val, chosen_dt, chosen_delta = None, None, 99999
        for e in elems:
            cref = e.get("contextRef") or e.get("contextref")
            if not cref or cref not in contexts: 
                continue
            ctype, dt, delta = contexts[cref]
            if want=="instant" and ctype!="instant":
                continue
            if want=="duration" and ctype!="duration":
                continue

            txt = (e.text or "").strip()
            if not txt: continue
            neg = txt.startswith("(") and txt.endswith(")")
            num = re.sub(r"[^\d\.\-]", "", txt)
            try:
                val = float(num)
                if neg: val = -val
            except: continue

            if want=="duration":
                if form_type=="10-Q" and 60 <= delta <= 120:  # quarter
                    if chosen_dt is None or dt > chosen_dt:
                        chosen_val, chosen_dt, chosen_delta = val, dt, delta
                elif form_type=="10-K" and 330 <= delta <= 370:  # year
                    if chosen_dt is None or dt > chosen_dt:
                        chosen_val, chosen_dt, chosen_delta = val, dt, delta
            else:
                if chosen_dt is None or dt > chosen_dt:
                    chosen_val, chosen_dt, chosen_delta = val, dt, delta
        return chosen_val, chosen_dt

    out = {}
    rev, rev_dt = None, None
    for tag in USGAAP_TAGS["revenue"]:
        v, dt = best_fact(tag, "duration")
        if v is not None:
            rev, rev_dt = v, dt
            break
    out["revenue"] = rev
    out["period_end"] = rev_dt.strftime("%Y-%m-%d") if rev_dt else None

    ni, ni_dt = best_fact("NetIncomeLoss","duration")
    out["net_income"] = ni
    if not out.get("period_end") and ni_dt:
        out["period_end"] = ni_dt.strftime("%Y-%m-%d")

    a, _ = best_fact("Assets","instant")
    l, _ = best_fact("Liabilities","instant")
    out["assets"] = a
    out["liabilities"] = l

    eq, _ = None, None
    for tag in USGAAP_TAGS["equity"]:
        v, _ = best_fact(tag,"instant")
        if v is not None:
            eq = v
            break
    out["equity"] = eq
    return out

# ------------------- MAIN -------------------
def main():
    filings = merge_sources(CIK)
    filings = filings[(filings["filed"] >= DATE_FROM) & (filings["filed"] <= DATE_TO)].copy()

    rows = []
    for _, r in filings.iterrows():
        index_url = str(r["primary"])
        if not index_url.startswith("http"):
            index_url = urljoin("https://www.sec.gov/Archives/", index_url)
        try:
            label, root = find_xbrl_instance_and_root(index_url)
            metrics = parse_xbrl_root(root, r["form"])
            rows.append({
                "filing_date": r["filed"].strftime("%Y-%m-%d"),
                "period_end": metrics.get("period_end"),
                "form": r["form"],
                "revenue": metrics.get("revenue"),
                "net_income": metrics.get("net_income"),
                "total_assets": metrics.get("assets"),
                "total_liabilities": metrics.get("liabilities"),
                "shareholders_equity": metrics.get("equity"),
                "xbrl_source": label
            })
            print(f"[OK] {r['filed'].date()} {r['form']} -> {metrics.get('period_end')}")
        except Exception as e:
            print(f"[ERR] {r['filed'].date()} {r['form']}: {e}")
            rows.append({
                "filing_date": r["filed"].strftime("%Y-%m-%d"),
                "period_end": None,
                "form": r["form"],
                "revenue": None,"net_income":None,
                "total_assets":None,"total_liabilities":None,"shareholders_equity":None,
                "xbrl_source": None
            })
        time.sleep(PAUSE)

    out = pd.DataFrame(rows).sort_values(["filing_date","form"])
    out.to_csv("tesla_financials_2009_2014.csv", index=False)
    print(f"[DONE] Saved {len(out)} rows to apple_financials_2009_2014_with_period.csv")

if __name__=="__main__":
    main()


[ERR] 2010-08-13 10-Q: No usable XBRL instance found
[ERR] 2010-11-12 10-Q: No usable XBRL instance found
[ERR] 2011-03-03 10-K: No usable XBRL instance found
[ERR] 2011-05-13 10-Q: No usable XBRL instance found
[OK] 2011-08-12 10-Q -> 2011-06-30
[OK] 2011-11-14 10-Q -> 2011-09-30
[OK] 2012-02-27 10-K -> 2011-12-31
[OK] 2012-03-28 10-K -> 2011-12-31
[OK] 2012-05-10 10-Q -> 2012-03-31
[OK] 2012-08-02 10-Q -> 2012-06-30
[OK] 2012-11-07 10-Q -> 2012-09-30
[OK] 2013-03-07 10-K -> 2012-12-31
[OK] 2013-05-10 10-Q -> 2013-03-31
[OK] 2013-08-09 10-Q -> 2013-06-30
[OK] 2013-11-08 10-Q -> 2013-09-30
[OK] 2014-02-26 10-K -> 2013-12-31
[OK] 2014-05-09 10-Q -> 2014-03-31
[OK] 2014-08-08 10-Q -> 2014-06-30
[OK] 2014-11-07 10-Q -> 2014-09-30
[OK] 2015-02-26 10-K -> 2014-12-31
[OK] 2015-05-11 10-Q -> 2015-03-31
[OK] 2015-08-07 10-Q -> 2015-06-30
[OK] 2015-11-05 10-Q -> 2015-09-30
[OK] 2016-02-24 10-K -> 2015-12-31
[OK] 2016-05-10 10-Q -> 2016-03-31
[OK] 2016-08-05 10-Q -> 2016-06-30
[OK] 2016-11-02 10